# Text Generation: From Bigrams to Transformers

This comprehensive notebook explores text generation through progressively complex approaches, from simple statistical models to modern neural architectures.

In [ ]:
# Configuration dictionary
# All hyperparameters and settings for the notebook are defined here
CONFIG = {
    # General settings
    'seed': 42,
    
    # Dataset settings
    'dataset_id': 'names',
    'train_split': 0.9,
    'val_split': 0.1,
    
    # Bigram model
    'bigram_smoothing': 1,  # Additive smoothing constant
    
    # MLP model
    'mlp_block_size': 3,           # Number of context characters
    'mlp_embedding_dim': 32,       # Embedding dimension
    'mlp_hidden_size': 128,        # Hidden layer size
    'mlp_learning_rate': 0.01,     # Learning rate
    'mlp_num_epochs': 1,           # Number of training epochs
    'mlp_batch_size': 1024,        # Batch size
    
    # RNN model
    'rnn_embedding_dim': 32,       # Embedding dimension
    'rnn_hidden_size': 128,        # Hidden state size
    'rnn_num_layers': 1,           # Number of RNN layers
    'rnn_learning_rate': 0.01,     # Learning rate
    'rnn_num_epochs': 1,           # Number of training epochs
    'rnn_batch_size': 128,         # Batch size
    
    # LSTM model
    'lstm_embedding_dim': 64,      # Embedding dimension
    'lstm_hidden_size': 256,       # Hidden state size
    'lstm_num_layers': 2,          # Number of LSTM layers
    'lstm_dropout': 0.2,           # Dropout probability
    'lstm_learning_rate': 0.001,   # Learning rate
    'lstm_num_epochs': 1,          # Number of training epochs
    'lstm_gradient_clip': 1.0,     # Gradient clipping max norm
    
    # Sampling settings
    'sample_max_length': 20,       # Maximum generation length
    'sample_temperature': 1.0,     # Default sampling temperature
    'sample_top_k': 5,             # Top-k sampling parameter
    'sample_top_p': 0.9,           # Nucleus sampling parameter
    
    # Analysis settings
    'novelty_num_samples': 1000,   # Number of samples for novelty analysis
    'temp_analysis_samples': 500,  # Samples for temperature analysis
    
    # Transformer (demo)
    'transformer_embed_size': 64,  # Embedding size
    'transformer_num_heads': 4,    # Number of attention heads
    'transformer_ff_size': 256,    # Feed-forward hidden size
    'transformer_dropout': 0.1,    # Dropout probability
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Learning Objectives

By the end of this notebook, you will understand:

1. **Language Modeling Fundamentals**: What language models are and how they work
2. **Character-Level Generation**: Building models that generate text one character at a time
3. **Statistical Approaches**: Using bigram frequencies to create probability distributions
4. **Neural Language Models**: Training networks to learn text patterns from data
5. **Recurrent Architectures**: RNN, LSTM, and GRU for sequential data
6. **Sampling Strategies**: Temperature, top-k, and nucleus sampling
7. **Model Evaluation**: Measuring quality through perplexity and novelty
8. **Modern Architectures**: Introduction to Transformers and GPT

## Part 1: Introduction to Text Generation

### What is Text Generation?

**Text generation** is the task of creating new text that follows patterns learned from training data. It's one of the most exciting applications of machine learning, powering:

- **Chatbots and assistants** (ChatGPT, Claude, Gemini)
- **Code completion** (GitHub Copilot, Cursor)
- **Creative writing** (story generation, poetry)
- **Translation and summarization**
- **Autocomplete** (email, search)

### How Does It Work?

At its core, text generation is about **predicting what comes next**. Given some text (called the *context*), a language model estimates the probability distribution over all possible next characters or words.

For example, given "hello wor", the model should assign high probability to "l" and "d" (completing "world").

### Autoregressive Generation

Most text generation is **autoregressive**: we generate one token at a time, using previously generated tokens as context for the next prediction.

```
Start: "."
Generate 1st char: "." → "a" (sample from P(char | "."))
Generate 2nd char: ".a" → "l" (sample from P(char | ".a"))
Generate 3rd char: ".al" → "i" (sample from P(char | ".al"))
...
Generate nth char: ".ali" → "." (end token)
Result: "ali"
```

### Reflection Questions

Before we dive into implementation, consider:

1. What makes generated text "good"? Should it be grammatically correct? Creative? Diverse?
2. How much context do you think is needed to predict the next character? The next word?
3. What are the risks of text generation systems? (bias, misinformation, etc.)

## Part 2: Setup and Data Exploration

Let's start by loading our dataset and understanding its structure.

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Import shared utilities from our local package
from aiml_notebooks import create_dataset, create_dataloaders, CharacterTokenizer, get_device

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch version: {torch.__version__}")

Run the following analysis.

In [ ]:
# Configuration
SEED = CONFIG['seed']
torch.manual_seed(SEED)

# Set device (MPS-compatible notebook)
device = get_device()


Display the output.

In [ ]:
# Load the names dataset using our factory function
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id=CONFIG['dataset_id'],
    splits=[CONFIG['train_split'], CONFIG['val_split']]
)

# Extract tokenizer and names
tokenizer = full_dataset.tokenizer
names = full_dataset.get_texts()

print(f"Total names: {len(names)}")
print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"\nFirst 10 names: {names[:10]}")

### Dataset Statistics

Let's explore our dataset to understand its characteristics.

In [ ]:
# Calculate statistics
lengths = [len(name) for name in names]
char_counts = Counter(''.join(names))

print(f"Name length statistics:")
print(f"  Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.1f}")
print(f"\nMost common characters:")
for char, count in char_counts.most_common(10):
    print(f"  '{char}': {count}")

Plot a histogram to understand the distribution.

In [ ]:
# Visualize name length distribution
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=range(min(lengths), max(lengths) + 2), edgecolor='black', alpha=0.7)
plt.xlabel('Name Length')
plt.ylabel('Frequency')
plt.title('Distribution of Name Lengths')
plt.grid(True, alpha=0.3)
plt.show()

### Understanding Tokenization

Our `CharacterTokenizer` converts characters to integers and vice versa. It also uses a special token (`.`) to mark the start and end of sequences.

In [ ]:
# Demonstrate tokenization
example_name = "alice"
encoded = tokenizer.encode(f".{example_name}.")
decoded = tokenizer.decode(encoded)

print(f"Original: {example_name}")
print(f"With special tokens: .{example_name}.")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")
print(f"\nVocabulary: {''.join(tokenizer.chars)}")

## Part 3: Language Modeling Fundamentals

### What is a Language Model?

A **language model** estimates the probability distribution over sequences of tokens. More specifically:

$$P(\text{sequence}) = P(c_1, c_2, ..., c_n)$$

Using the chain rule of probability:

$$P(c_1, c_2, ..., c_n) = P(c_1) \cdot P(c_2 | c_1) \cdot P(c_3 | c_1, c_2) \cdot ... \cdot P(c_n | c_1, ..., c_{n-1})$$

The challenge is estimating these conditional probabilities efficiently.

### N-gram Models

The simplest approach is to make a **Markov assumption**: the next token depends only on the previous N tokens.

- **Unigram** (N=0): $P(c_i)$ - each character is independent
- **Bigram** (N=1): $P(c_i | c_{i-1})$ - depends on previous character
- **Trigram** (N=2): $P(c_i | c_{i-2}, c_{i-1})$ - depends on two previous characters

We'll start with a bigram model.

## Part 4: Bigram Statistical Model

### Theory: Counting Bigrams

A **bigram** is a sequence of two consecutive characters. To build a bigram model:

1. **Count** all bigrams in the training data
2. **Normalize** counts to create probability distributions
3. **Sample** from these distributions to generate new text

For example, if we see "hello" in the data, we count:
- (h, e), (e, l), (l, l), (l, o)

The probability $P(\text{e} | \text{h})$ is:

$$P(e | h) = \frac{\text{count}(h, e)}{\sum_{c'} \text{count}(h, c')}$$

In [ ]:
# Count bigrams in our dataset
EOS_CHAR = '.'
bigram_counts = {}

for name in names:
    # Add special tokens for start and end
    chars = [EOS_CHAR] + list(name) + [EOS_CHAR]
    
    # Count each bigram
    for c1, c2 in zip(chars, chars[1:]):
        bigram = (c1, c2)
        bigram_counts[bigram] = bigram_counts.get(bigram, 0) + 1

print(f"Total unique bigrams: {len(bigram_counts)}")
print(f"\nMost common bigrams:")
for bigram, count in sorted(bigram_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  '{bigram[0]}{bigram[1]}': {count}")

### Visualizing Bigram Frequencies

Let's create a matrix where rows represent the first character and columns represent the second character.

In [ ]:
# Create character-to-index mapping
stoi = tokenizer.char_to_idx
itos = tokenizer.idx_to_char
vocab_size = tokenizer.vocab_size

# Create bigram frequency matrix
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

for (c1, c2), count in bigram_counts.items():
    i1, i2 = stoi[c1], stoi[c2]
    N[i1, i2] = count

print(f"Bigram matrix shape: {N.shape}")
print(f"Total bigram occurrences: {N.sum().item()}")

Create a bar chart to compare values.

In [ ]:
# Visualize the bigram matrix
plt.figure(figsize=(12, 10))
plt.imshow(N, cmap='Blues')
plt.colorbar(label='Frequency')
plt.xlabel('Second Character')
plt.ylabel('First Character')
plt.title('Bigram Frequency Matrix')
plt.show()

### Converting Counts to Probabilities

To use our bigram counts for generation, we need to normalize each row into a probability distribution.

In [ ]:
# Add smoothing to avoid zero probabilities
# (This is called "additive smoothing" or "Laplace smoothing")
P = (N + CONFIG['bigram_smoothing']).float()  # Add smoothing constant
P = P / P.sum(dim=1, keepdim=True)  # Normalize each row

# Verify it's a valid probability distribution
print(f"Shape: {P.shape}")
print(f"All rows sum to 1.0: {torch.allclose(P.sum(dim=1), torch.ones(vocab_size))}")
print(f"\nExample: probability of characters following '.' (start):")
start_idx = stoi[EOS_CHAR]
for i, prob in enumerate(P[start_idx][:10]):
    print(f"  P('{itos[i]}' | '.'): {prob:.4f}")

### Sampling from the Model

Now we can generate new names by sampling from our probability distributions.

In [ ]:
def sample_bigram_model(P, stoi, itos, max_length=CONFIG['sample_max_length'], seed=None):
    """Generate a name using the bigram model."""
    if seed is not None:
        generator = torch.Generator().manual_seed(seed)
    else:
        generator = None
    
    current_idx = stoi[EOS_CHAR]  # Start with special token
    name = []
    
    for _ in range(max_length):
        # Get probability distribution for next character
        probs = P[current_idx]
        
        # Sample next character
        next_idx = torch.multinomial(probs, num_samples=1, generator=generator).item()
        next_char = itos[next_idx]
        
        # Stop if we hit the end token
        if next_char == EOS_CHAR:
            break
        
        name.append(next_char)
        current_idx = next_idx
    
    return ''.join(name)

# Generate 20 sample names
print("Generated names from bigram model:")
generator = torch.Generator().manual_seed(SEED)
for _ in range(20):
    name = sample_bigram_model(P, stoi, itos, seed=generator.initial_seed())
    generator = generator.manual_seed(generator.initial_seed() + 1)
    print(f"  {name.capitalize()}")

### Reflection: Quality of Bigram Models

Notice that the generated names often:
- Sound somewhat name-like
- Follow basic phonetic patterns
- Sometimes produce gibberish or very short names

**Why?** The bigram model only looks at one previous character, so it can't capture longer-range patterns. For example, it doesn't know that names rarely repeat the same syllable multiple times.

**Question**: How might we improve this model?

## Part 5: Evaluating Language Models

### Loss Functions: Likelihood and Cross-Entropy

To train neural models, we need a loss function that measures how well our model predicts the training data.

The **likelihood** of a sequence is the product of probabilities:

$$L = \prod_i P(c_i | \text{context}_i)$$

The **log-likelihood** is easier to work with:

$$\log L = \sum_i \log P(c_i | \text{context}_i)$$

We typically use **negative log-likelihood** (NLL) as our loss (lower is better):

$$\text{Loss} = -\frac{1}{N} \sum_i \log P(c_i | \text{context}_i)$$

This is equivalent to **cross-entropy loss**, which PyTorch provides.

In [ ]:
# Calculate the loss of our bigram model on the training data
total_log_likelihood = 0.0
num_bigrams = 0

for name in names:
    chars = [EOS_CHAR] + list(name) + [EOS_CHAR]
    for c1, c2 in zip(chars, chars[1:]):
        i1, i2 = stoi[c1], stoi[c2]
        prob = P[i1, i2]
        total_log_likelihood += torch.log(prob)
        num_bigrams += 1

avg_nll = -total_log_likelihood / num_bigrams
perplexity = torch.exp(avg_nll)

print(f"Bigram model evaluation:")
print(f"  Average negative log-likelihood: {avg_nll:.4f}")
print(f"  Perplexity: {perplexity:.4f}")

### Perplexity

**Perplexity** is an intuitive metric for language models:

$$\text{Perplexity} = e^{\text{NLL}}$$

It can be interpreted as the "effective vocabulary size" - how many equally likely choices the model faces at each step.

- Lower perplexity = better model
- Perplexity of 1 = perfect predictions
- Perplexity equal to vocab size = random guessing

## Part 6: Neural Bigram Model

### From Statistics to Learning

Instead of manually counting bigrams, can we train a neural network to learn the same probability distribution?

**Architecture**:
- Input: One-hot encoded character (size: vocab_size)
- Layer: Single linear transformation (weights matrix W)
- Output: Logits for next character (size: vocab_size)
- Loss: Cross-entropy

The network will learn weights that encode the bigram frequencies!

### Understanding One-Hot Encoding

We convert each character to a vector where all elements are 0 except one position:

```
'a' → [0, 1, 0, 0, ..., 0]
'b' → [0, 0, 1, 0, ..., 0]
```

This prevents the model from treating characters as having numerical relationships (e.g., 'z' > 'a').

In [ ]:
# Create training dataset (X: current char, Y: next char)
xs, ys = [], []

for name in names:
    chars = [EOS_CHAR] + list(name) + [EOS_CHAR]
    for c1, c2 in zip(chars, chars[1:]):
        xs.append(stoi[c1])
        ys.append(stoi[c2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(f"Training examples: {len(xs)}")
print(f"First 5 examples:")
for i in range(5):
    print(f"  {itos[xs[i].item()]} → {itos[ys[i].item()]}")

Display the output.

In [ ]:
# One-hot encode the inputs
xs_onehot = F.one_hot(xs, num_classes=vocab_size).float()

print(f"One-hot encoded shape: {xs_onehot.shape}")
print(f"\nExample: character '{itos[xs[0].item()]}'")
print(f"One-hot vector: {xs_onehot[0]}")

Create a bar chart to compare values.

In [ ]:
# Visualize one-hot encoding
plt.figure(figsize=(12, 4))
plt.imshow(xs_onehot[:30].T, cmap='Blues', aspect='auto')
plt.xlabel('Training Example')
plt.ylabel('Character Index')
plt.title('One-Hot Encoded Characters (first 30 examples)')
plt.colorbar()
plt.show()

### Training the Neural Bigram Model

Now let's train a simple neural network to learn the bigram probabilities.

In [ ]:
# Initialize weights randomly
generator = torch.Generator().manual_seed(SEED)
W = torch.randn((vocab_size, vocab_size), generator=generator, requires_grad=True)

print(f"Weights shape: {W.shape}")
print(f"Trainable parameters: {W.numel()}")

Train the model and monitor progress.

In [ ]:
# Training loop
learning_rate = 50
num_epochs = 1

losses = []

for epoch in range(num_epochs):
    # Forward pass
    logits = xs_onehot @ W
    
    # Calculate loss
    loss = F.cross_entropy(logits, ys)
    
    # Backward pass
    W.grad = None
    loss.backward()
    
    # Update weights
    W.data -= learning_rate * W.grad
    
    losses.append(loss.item())
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}, Loss: {loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

Visualize the results.

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss (Negative Log-Likelihood)')
plt.title('Training Curve: Neural Bigram Model')
plt.grid(True, alpha=0.3)
plt.show()

### What Did the Network Learn?

Let's compare the learned weights with our manual bigram probabilities.

In [ ]:
# Convert learned weights to probabilities
W_probs = W.exp()
W_probs = W_probs / W_probs.sum(dim=1, keepdim=True)

# Compare with manual model
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(W_probs.detach().numpy(), cmap='Blues')
axes[0].set_title('Learned Weights (Neural Model)')
axes[0].set_xlabel('Next Character')
axes[0].set_ylabel('Current Character')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(P.numpy(), cmap='Blues')
axes[1].set_title('Counted Probabilities (Statistical Model)')
axes[1].set_xlabel('Next Character')
axes[1].set_ylabel('Current Character')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("The learned weights closely match the statistical model!")

### Reflection: Why Use Neural Networks?

For bigrams, counting is simpler than neural networks. But as we increase context length:

- **Trigrams**: 27³ = 19,683 possible combinations
- **4-grams**: 27⁴ = 531,441 combinations

Counting becomes impractical! Neural networks can:
1. Generalize from similar contexts
2. Handle variable-length context
3. Learn distributed representations

This is why we use neural language models for longer contexts.

## Part 7: MLP Language Model

### Theory: Fixed-Context Neural Model

Instead of looking at just one previous character, let's use multiple characters of context.

**Architecture**:
1. Input: Last N characters (e.g., N=3)
2. Embedding: Map each character to a learned vector
3. Hidden layer: Learn patterns from concatenated embeddings
4. Output: Predict next character

This is similar to the approach used in [Bengio et al. 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf).

In [ ]:
# Create fixed-context dataset
block_size = CONFIG['mlp_block_size']  # Use config value

def build_context_dataset(names, stoi, block_size):
    X, Y = [], []
    
    for name in names:
        chars = [EOS_CHAR] * block_size + list(name) + [EOS_CHAR]
        for i in range(len(chars) - block_size):
            context = chars[i:i+block_size]
            target = chars[i+block_size]
            X.append([stoi[c] for c in context])
            Y.append(stoi[target])
    
    return torch.tensor(X), torch.tensor(Y)

X_mlp, Y_mlp = build_context_dataset(names, stoi, block_size)

print(f"MLP dataset size: {len(X_mlp)}")
print(f"Context length: {block_size}")
print(f"\nFirst 5 examples:")
for i in range(5):
    context_chars = ''.join([itos[idx.item()] for idx in X_mlp[i]])
    target_char = itos[Y_mlp[i].item()]
    print(f"  '{context_chars}' → '{target_char}'")

Display the output.

In [ ]:
model_mlp = MLPLanguageModel(
    vocab_size, 
    block_size,
    embedding_dim=CONFIG['mlp_embedding_dim'],
    hidden_size=CONFIG['mlp_hidden_size']
)
num_params = sum(p.numel() for p in model_mlp.parameters())
print(f"MLP Model: {num_params} parameters")

Train the model and monitor progress.

In [ ]:
# Training the MLP model
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=CONFIG['mlp_learning_rate'])
num_epochs = CONFIG['mlp_num_epochs']
batch_size = CONFIG['mlp_batch_size']

mlp_losses = []

for epoch in range(num_epochs):
    # Mini-batch training
    indices = torch.randperm(len(X_mlp))
    epoch_loss = 0.0
    num_batches = 0
    
    for i in range(0, len(X_mlp), batch_size):
        batch_indices = indices[i:i+batch_size]
        X_batch = X_mlp[batch_indices]
        Y_batch = Y_mlp[batch_indices]
        
        # Forward pass
        logits = model_mlp(X_batch)
        loss = F.cross_entropy(logits, Y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    mlp_losses.append(avg_loss)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d}, Loss: {avg_loss:.4f}")

print(f"\nFinal loss: {mlp_losses[-1]:.4f}")

Visualize the results.

In [ ]:
# Plot MLP training curve
plt.figure(figsize=(10, 4))
plt.plot(mlp_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curve: MLP Language Model')
plt.grid(True, alpha=0.3)
plt.show()

Evaluate the model on the test set.

In [ ]:
# Generate samples from MLP model
@torch.no_grad()
def sample_mlp_model(model, stoi, itos, block_size, max_length=CONFIG['sample_max_length'], temperature=1.0):
    model.eval()
    context = [stoi[EOS_CHAR]] * block_size
    name = []
    
    for _ in range(max_length):
        x = torch.tensor([context])
        logits = model(x)
        probs = F.softmax(logits[0] / temperature, dim=0)
        next_idx = torch.multinomial(probs, 1).item()
        next_char = itos[next_idx]
        
        if next_char == EOS_CHAR:
            break
        
        name.append(next_char)
        context = context[1:] + [next_idx]
    
    return ''.join(name)

print("Generated names from MLP model:")
for _ in range(20):
    name = sample_mlp_model(model_mlp, stoi, itos, block_size)
    print(f"  {name.capitalize()}")

### Comparison: Bigram vs MLP

Notice how the MLP-generated names:
- Sound more natural
- Have better syllable structure
- Are less repetitive

This is because the model sees 3 characters of context instead of just 1!

## Part 8: Recurrent Neural Networks (RNNs)

### Theory: Variable-Length Context

The MLP model has a fixed context window. What if we want to consider the entire history?

**Recurrent Neural Networks** (RNNs) maintain a hidden state that accumulates information over time:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

Benefits:
- Handle sequences of any length
- Share parameters across time steps
- Maintain context in hidden state

Challenges:
- Vanishing/exploding gradients
- Difficulty learning long-range dependencies

In [ ]:
model_rnn = RNNLanguageModel(
    vocab_size,
    embedding_dim=CONFIG['rnn_embedding_dim'],
    hidden_size=CONFIG['rnn_hidden_size'],
    num_layers=CONFIG['rnn_num_layers']
)
num_params = sum(p.numel() for p in model_rnn.parameters())
print(f"RNN Model: {num_params} parameters")

Prepare the data loaders for training and validation.

In [ ]:
# Create DataLoader for RNN training
train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=CONFIG['rnn_batch_size']
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Train the model and monitor progress.

In [ ]:
# Training the RNN model
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=CONFIG['rnn_learning_rate'])
num_epochs = CONFIG['rnn_num_epochs']

rnn_train_losses = []
rnn_val_losses = []

for epoch in range(num_epochs):
    # Training
    model_rnn.train()
    train_loss = 0.0
    for X_batch, Y_batch in train_loader:
        logits, _ = model_rnn(X_batch)
        loss = F.cross_entropy(logits.view(-1, vocab_size), Y_batch.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model_rnn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            logits, _ = model_rnn(X_batch)
            loss = F.cross_entropy(logits.view(-1, vocab_size), Y_batch.view(-1))
            val_loss += loss.item()
    
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    rnn_train_losses.append(train_loss)
    rnn_val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1:2d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

Visualize the results.

In [ ]:
# Plot RNN training curves
plt.figure(figsize=(10, 4))
plt.plot(rnn_train_losses, label='Train')
plt.plot(rnn_val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves: RNN Language Model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Evaluate the model on the test set.

In [ ]:
# Generate samples from RNN model
@torch.no_grad()
def sample_rnn_model(model, stoi, itos, max_length=CONFIG['sample_max_length'], temperature=1.0):
    model.eval()
    current_idx = stoi[EOS_CHAR]
    name = []
    hidden = None
    
    for _ in range(max_length):
        x = torch.tensor([[current_idx]])
        logits, hidden = model(x, hidden)
        probs = F.softmax(logits[0, -1] / temperature, dim=0)
        next_idx = torch.multinomial(probs, 1).item()
        next_char = itos[next_idx]
        
        if next_char == EOS_CHAR:
            break
        
        name.append(next_char)
        current_idx = next_idx
    
    return ''.join(name)

print("Generated names from RNN model:")
for _ in range(20):
    name = sample_rnn_model(model_rnn, stoi, itos)
    print(f"  {name.capitalize()}")

## Part 9: LSTM and GRU

### Theory: Advanced Recurrent Cells

Standard RNNs struggle with long-range dependencies due to vanishing gradients. Two popular solutions:

**LSTM (Long Short-Term Memory)**:
- Maintains a separate cell state
- Uses gates to control information flow: forget gate, input gate, output gate
- Better at capturing long-range dependencies

**GRU (Gated Recurrent Unit)**:
- Simplified version of LSTM
- Uses update and reset gates
- Fewer parameters, often performs similarly to LSTM

In [ ]:
model_lstm = LSTMLanguageModel(
    vocab_size,
    embedding_dim=CONFIG['lstm_embedding_dim'],
    hidden_size=CONFIG['lstm_hidden_size'],
    num_layers=CONFIG['lstm_num_layers'],
    dropout=CONFIG['lstm_dropout']
)
num_params = sum(p.numel() for p in model_lstm.parameters())
print(f"LSTM Model: {num_params} parameters")

Train the model and monitor progress.

In [ ]:
# Train LSTM (same training loop as RNN)
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=CONFIG['lstm_learning_rate'])
num_epochs = CONFIG['lstm_num_epochs']

lstm_train_losses = []
lstm_val_losses = []

for epoch in range(num_epochs):
    # Training
    model_lstm.train()
    train_loss = 0.0
    for X_batch, Y_batch in train_loader:
        logits, _ = model_lstm(X_batch)
        loss = F.cross_entropy(logits.view(-1, vocab_size), Y_batch.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), CONFIG['lstm_gradient_clip'])
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model_lstm.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            logits, _ = model_lstm(X_batch)
            loss = F.cross_entropy(logits.view(-1, vocab_size), Y_batch.view(-1))
            val_loss += loss.item()
    
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    lstm_train_losses.append(train_loss)
    lstm_val_losses.append(val_loss)
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch+1:2d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print(f"\nFinal - Train: {lstm_train_losses[-1]:.4f}, Val: {lstm_val_losses[-1]:.4f}")

Visualize the results with multiple plots for comparison.

In [ ]:
# Compare training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(rnn_train_losses, label='RNN', alpha=0.7)
axes[0].plot(lstm_train_losses, label='LSTM', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(rnn_val_losses, label='RNN', alpha=0.7)
axes[1].plot(lstm_val_losses, label='LSTM', alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Display the output.

In [ ]:
# Generate samples from LSTM model
print("Generated names from LSTM model:")
for _ in range(20):
    name = sample_rnn_model(model_lstm, stoi, itos)  # Same sampling function works
    print(f"  {name.capitalize()}")

## Part 10: Sampling Strategies

### Theory: Controlling Generation Quality

How we sample from the probability distribution greatly affects the quality and diversity of generated text.

**Strategies**:

1. **Greedy Sampling**: Always pick the most likely token
   - Pros: Deterministic, high quality
   - Cons: Repetitive, boring

2. **Temperature Sampling**: Scale logits before softmax
   - T < 1: More conservative (sharper distribution)
   - T = 1: Use model's distribution
   - T > 1: More creative (flatter distribution)

3. **Top-k Sampling**: Only sample from top k most likely tokens
   - Prevents sampling from very unlikely tokens
   - Fixed cutoff regardless of distribution

4. **Nucleus (Top-p) Sampling**: Sample from smallest set with cumulative probability ≥ p
   - Adapts to distribution shape
   - Better than top-k for most applications

In [ ]:
# Demonstrate temperature effects
def visualize_temperature(logits, temperatures=[0.5, 1.0, 1.5, 2.0]):
    fig, axes = plt.subplots(1, len(temperatures), figsize=(16, 3))
    
    for i, temp in enumerate(temperatures):
        probs = F.softmax(logits / temp, dim=0)
        axes[i].bar(range(len(probs)), probs.detach().numpy())
        axes[i].set_title(f'Temperature = {temp}')
        axes[i].set_xlabel('Token Index')
        axes[i].set_ylabel('Probability')
    
    plt.tight_layout()
    plt.show()

# Create example logits
example_logits = torch.randn(vocab_size) * 2
visualize_temperature(example_logits)

Run the following analysis.

In [ ]:
# Implement different sampling strategies
def sample_with_strategy(logits, strategy='greedy', temperature=1.0, top_k=0, top_p=0.0):
    """Sample next token using specified strategy."""
    
    if strategy == 'greedy':
        return torch.argmax(logits).item()
    
    # Apply temperature
    logits = logits / temperature
    
    if strategy == 'temperature':
        probs = F.softmax(logits, dim=0)
        return torch.multinomial(probs, 1).item()
    
    elif strategy == 'top_k':
        # Keep only top k logits
        top_k_logits, top_k_indices = torch.topk(logits, top_k)
        probs = F.softmax(top_k_logits, dim=0)
        sampled_idx = torch.multinomial(probs, 1).item()
        return top_k_indices[sampled_idx].item()
    
    elif strategy == 'nucleus':
        # Sort logits in descending order
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        sorted_probs = F.softmax(sorted_logits, dim=0)
        
        # Calculate cumulative probabilities
        cumsum_probs = torch.cumsum(sorted_probs, dim=0)
        
        # Find cutoff index
        cutoff_idx = torch.where(cumsum_probs > top_p)[0]
        if len(cutoff_idx) > 0:
            cutoff_idx = cutoff_idx[0].item() + 1
        else:
            cutoff_idx = len(sorted_probs)
        
        # Sample from nucleus
        nucleus_probs = sorted_probs[:cutoff_idx]
        nucleus_probs = nucleus_probs / nucleus_probs.sum()
        sampled_idx = torch.multinomial(nucleus_probs, 1).item()
        return sorted_indices[sampled_idx].item()
    
    raise ValueError(f"Unknown strategy: {strategy}")

Evaluate the model on the test set.

In [ ]:
# Compare sampling strategies
@torch.no_grad()
def sample_rnn_with_strategy(model, stoi, itos, strategy='temperature', **kwargs):
    model.eval()
    current_idx = stoi[EOS_CHAR]
    name = []
    hidden = None
    
    for _ in range(20):
        x = torch.tensor([[current_idx]])
        logits, hidden = model(x, hidden)
        next_idx = sample_with_strategy(logits[0, -1], strategy=strategy, **kwargs)
        next_char = itos[next_idx]
        
        if next_char == EOS_CHAR:
            break
        
        name.append(next_char)
        current_idx = next_idx
    
    return ''.join(name)

# Generate with different strategies
print("Greedy Sampling:")
for _ in range(10):
    name = sample_rnn_with_strategy(model_lstm, stoi, itos, strategy='greedy')
    print(f"  {name.capitalize()}")

print("\nTemperature=0.5 (Conservative):")
for _ in range(10):
    name = sample_rnn_with_strategy(model_lstm, stoi, itos, strategy='temperature', temperature=0.5)
    print(f"  {name.capitalize()}")

print("\nTemperature=1.5 (Creative):")
for _ in range(10):
    name = sample_rnn_with_strategy(model_lstm, stoi, itos, strategy='temperature', temperature=1.5)
    print(f"  {name.capitalize()}")

print("\nTop-k=5:")
for _ in range(10):
    name = sample_rnn_with_strategy(model_lstm, stoi, itos, strategy='top_k', top_k=5)
    print(f"  {name.capitalize()}")

print("\nNucleus (top-p=0.9):")
for _ in range(10):
    name = sample_rnn_with_strategy(model_lstm, stoi, itos, strategy='nucleus', top_p=0.9)
    print(f"  {name.capitalize()}")

### Reflection: Sampling Strategy Trade-offs

**Quality vs. Diversity**: There's a fundamental trade-off:
- Conservative sampling (low temperature, greedy) produces high-quality but repetitive outputs
- Creative sampling (high temperature, nucleus) produces diverse but potentially lower-quality outputs

**Question**: What sampling strategy would you use for:
1. A chatbot assistant?
2. A creative writing tool?
3. Code completion?

## Part 11: Analyzing Novelty vs Memorization

### Theory: Generation Quality Metrics

How do we know if our model is truly generating new text or just memorizing the training data?

**Metrics**:
1. **Novelty**: Percentage of generated samples not in training data
2. **Diversity**: Number of unique samples generated
3. **Quality**: Subjective assessment or perplexity on held-out data

Ideally, we want high novelty, high diversity, and high quality!

In [ ]:
# Analyze novelty
def analyze_novelty(model, stoi, itos, training_names, num_samples=CONFIG['novelty_num_samples'], temperature=1.0):
    """Generate samples and analyze novelty vs memorization."""
    training_set = set(training_names)
    generated = []
    
    for _ in range(num_samples):
        name = sample_rnn_model(model, stoi, itos, temperature=temperature)
        if name:  # Skip empty names
            generated.append(name)
    
    # Calculate metrics
    unique = set(generated)
    novel = [name for name in generated if name not in training_set]
    memorized = [name for name in generated if name in training_set]
    
    print(f"Analysis of {num_samples} generated names:")
    print(f"  Total generated: {len(generated)}")
    print(f"  Unique: {len(unique)} ({100*len(unique)/len(generated):.1f}%)")
    print(f"  Novel (not in training): {len(novel)} ({100*len(novel)/len(generated):.1f}%)")
    print(f"  Memorized (in training): {len(memorized)} ({100*len(memorized)/len(generated):.1f}%)")
    
    return {
        'generated': generated,
        'unique': unique,
        'novel': novel,
        'memorized': memorized
    }

lstm_analysis = analyze_novelty(model_lstm, stoi, itos, names, num_samples=CONFIG['novelty_num_samples'])

Display the output.

In [ ]:
# Show examples
print("\nExample novel names:")
for name in list(lstm_analysis['novel'])[:20]:
    print(f"  {name.capitalize()}")

print("\nExample memorized names:")
for name in list(lstm_analysis['memorized'])[:20]:
    print(f"  {name.capitalize()}")

Display the output.

In [ ]:
# Compare novelty across temperatures
temperatures = [0.5, 0.8, 1.0, 1.2, 1.5]
novelty_rates = []
diversity_rates = []

for temp in temperatures:
    analysis = analyze_novelty(model_lstm, stoi, itos, names, num_samples=CONFIG['temp_analysis_samples'], temperature=temp)
    novelty_rate = 100 * len(analysis['novel']) / len(analysis['generated'])
    diversity_rate = 100 * len(analysis['unique']) / len(analysis['generated'])
    novelty_rates.append(novelty_rate)
    diversity_rates.append(diversity_rate)
    print(f"\nTemperature {temp}: Novelty={novelty_rate:.1f}%, Diversity={diversity_rate:.1f}%")

Visualize the results.

In [ ]:
# Visualize temperature effects
plt.figure(figsize=(10, 4))
plt.plot(temperatures, novelty_rates, marker='o', label='Novelty')
plt.plot(temperatures, diversity_rates, marker='s', label='Diversity')
plt.xlabel('Temperature')
plt.ylabel('Percentage (%)')
plt.title('Effect of Temperature on Generation Metrics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Part 12: Introduction to Transformers

### Theory: Attention and Self-Attention

**Limitations of RNNs**:
- Sequential processing (can't parallelize)
- Limited context window in practice
- Gradient flow issues

**Transformers** solve these with **self-attention**:
- Each position attends to all other positions
- Fully parallelizable
- Direct connections between any two tokens

**Key Components**:
1. **Self-Attention**: Compute relationships between all tokens
2. **Positional Encoding**: Add position information
3. **Feed-Forward Networks**: Process attention outputs
4. **Layer Normalization**: Stabilize training

**GPT Architecture**: Decoder-only transformer for autoregressive generation

### Simplified Self-Attention Example

Let's implement a simple self-attention mechanism to understand how it works.

In [ ]:
# Simple self-attention implementation
class SelfAttention(nn.Module):
    def __init__(self, embed_size, head_size):
        super().__init__()
        self.key = nn.Linear(embed_size, head_size, bias=False)
        self.query = nn.Linear(embed_size, head_size, bias=False)
        self.value = nn.Linear(embed_size, head_size, bias=False)
    
    def forward(self, x):
        # x shape: (batch, seq_len, embed_size)
        B, T, C = x.shape
        
        # Compute queries, keys, values
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)
        
        # Compute attention scores
        scores = q @ k.transpose(-2, -1)  # (B, T, T)
        scores = scores / (k.shape[-1] ** 0.5)  # Scale
        
        # Apply causal mask (for autoregressive generation)
        tril = torch.tril(torch.ones(T, T))
        scores = scores.masked_fill(tril == 0, float('-inf'))
        
        # Apply softmax
        attn_weights = F.softmax(scores, dim=-1)  # (B, T, T)
        
        # Compute weighted values
        out = attn_weights @ v  # (B, T, head_size)
        
        return out, attn_weights

# Test self-attention
embed_size = 32
head_size = 16
seq_len = 10

attn = SelfAttention(embed_size, head_size)
test_input = torch.randn(1, seq_len, embed_size)
out, weights = attn(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention weights shape: {weights.shape}")

Create a bar chart to compare values.

In [ ]:
# Visualize attention weights
plt.figure(figsize=(8, 6))
plt.imshow(weights[0].detach().numpy(), cmap='Blues')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Self-Attention Weights (Causal Mask)')
plt.colorbar()
plt.show()

print("Notice the triangular pattern - each position can only attend to previous positions!")

### A Simple Transformer Block

Let's build a minimal transformer block for character-level generation.

In [ ]:
class SimpleTransformerBlock(nn.Module):
    def __init__(self, embed_size, num_heads, ff_size, dropout=0.1):
        super().__init__()
        head_size = embed_size // num_heads
        
        # Multi-head attention (simplified: just one head)
        self.attn = SelfAttention(embed_size, head_size)
        self.attn_proj = nn.Linear(head_size, embed_size)
        
        # Feed-forward network
        self.ff = nn.Sequential(
            nn.Linear(embed_size, ff_size),
            nn.ReLU(),
            nn.Linear(ff_size, embed_size),
            nn.Dropout(dropout)
        )
        
        # Layer normalization
        self.ln1 = nn.LayerNorm(embed_size)
        self.ln2 = nn.LayerNorm(embed_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Self-attention with residual connection
        attn_out, _ = self.attn(self.ln1(x))
        # Project back to embed_size
        attn_out = self.attn_proj(attn_out)
        x = x + self.dropout(attn_out)
        
        # Feed-forward with residual connection
        ff_out = self.ff(self.ln2(x))
        x = x + ff_out
        
        return x

# Test transformer block
block = SimpleTransformerBlock(embed_size=64, num_heads=4, ff_size=256)
test_input = torch.randn(1, 10, 64)
test_output = block(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")

### Why Transformers Dominate Modern NLP

**Advantages**:
1. **Parallelization**: Process entire sequence at once
2. **Long-range dependencies**: Direct connections between any tokens
3. **Scalability**: Performance improves with model size and data
4. **Interpretability**: Attention weights show what the model focuses on

**Examples**:
- **GPT** (Generative Pre-trained Transformer): Decoder-only for text generation
- **BERT**: Encoder-only for understanding
- **T5, BART**: Encoder-decoder for translation and summarization

Modern large language models (ChatGPT, Claude, etc.) are all based on the transformer architecture!

## Part 13: Summary and Best Practices

### What We Learned

1. **Language modeling** is about predicting probability distributions over sequences
2. **Bigram models** use statistics but have limited context
3. **Neural models** learn patterns from data and generalize better
4. **RNNs/LSTMs/GRUs** handle variable-length sequences with recurrent connections
5. **Sampling strategies** control the quality-diversity trade-off
6. **Transformers** use self-attention for better parallelization and long-range dependencies

### Model Comparison

| Model | Context | Parallelizable | Long-Range | Quality |
|-------|---------|----------------|------------|----------|
| Bigram | 1 char | ✓ | ✗ | Low |
| MLP | Fixed | ✓ | Limited | Medium |
| RNN | Variable | ✗ | Limited | Medium |
| LSTM | Variable | ✗ | Better | Good |
| Transformer | Variable | ✓ | Excellent | Excellent |

### Best Practices

**Training**:
- Use gradient clipping for RNNs (prevent exploding gradients)
- Add dropout for regularization
- Monitor validation loss to detect overfitting
- Use learning rate scheduling

**Generation**:
- Start with temperature=1.0, adjust based on needs
- Use nucleus sampling (top-p) for good quality-diversity balance
- Set max_length to prevent infinite loops
- Analyze novelty vs memorization

**Architecture**:
- For character-level: RNN/LSTM works well
- For word-level: Use transformers
- For production: Use pre-trained models (GPT, BERT, etc.)

### Common Issues and Solutions

| Problem | Solution |
|---------|----------|
| Repetitive outputs | Lower temperature, use nucleus sampling |
| Nonsensical outputs | Higher temperature, more training |
| Short sequences | Adjust special token probabilities |
| Memorization | Add dropout, more diverse training data |
| Slow training | Use transformers, smaller sequences |
| Exploding gradients | Gradient clipping, lower learning rate |

## Part 14: Further Exploration

### Experiments to Try

1. **Different Datasets**: Train on words, code, or other languages
2. **Deeper Models**: Add more layers and see how it affects quality
3. **Beam Search**: Implement beam search instead of sampling
4. **Byte-Pair Encoding**: Use subword tokenization instead of characters
5. **Fine-tuning**: Start with a pre-trained model and adapt it
6. **Conditional Generation**: Add control codes for style/length

### Resources for Learning More

**Courses**:
- [Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html) by Andrej Karpathy
- [Stanford CS224N](http://web.stanford.edu/class/cs224n/) - NLP with Deep Learning
- [Fast.ai](https://course.fast.ai/) - Practical Deep Learning

**Papers**:
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Original Transformer paper
- [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) - GPT-3
- [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751) - Nucleus sampling

**Libraries**:
- [Hugging Face Transformers](https://huggingface.co/transformers/) - Pre-trained models
- [PyTorch](https://pytorch.org/) - Deep learning framework
- [TextGeneration](https://github.com/huggingface/text-generation-inference) - Efficient inference

## Conclusion

Text generation has come a long way from simple n-gram models to sophisticated transformer architectures. The key insights:

1. **Context matters**: More context generally leads to better predictions
2. **Architecture matters**: Transformers outperform RNNs for most tasks
3. **Sampling matters**: How you sample is as important as the model itself
4. **Scale matters**: Larger models trained on more data tend to perform better

The field continues to evolve rapidly, with new architectures, training techniques, and applications emerging constantly. The fundamentals covered in this notebook provide a solid foundation for understanding and working with modern language models.

**Thank you for following along! Happy generating! 🎉**